In [1]:
import os
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
from tqdm.auto import tqdm


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

seed_everything(1234)
print("Device:", device)

Device: cuda


In [2]:
DATA_ROOT = "./data"

N_TRAIN = 1200
N_VAL = 300
N_TEST = 300

REPEATS = 20
EPOCHS = 100
BATCH_SIZE = 4

LR = 1e-3
BETA1 = 0.9
BETA2 = 0.999
EPS = 1e-7

BASE_SEED = 1234

KERNEL_SIZE = 4
NUM_FILTERS = 20


print(
    f"H1 MNIST | kernel={KERNEL_SIZE}x{KERNEL_SIZE}, "
    f"filters={NUM_FILTERS}, repeats={REPEATS}, epochs={EPOCHS}"
)

H1 MNIST | kernel=4x4, filters=20, repeats=20, epochs=100


In [3]:
train_full = MNIST(root=DATA_ROOT, train=True, download=True)
test_full = MNIST(root=DATA_ROOT, train=False, download=True)

x_train_full = train_full.data.clone().float().unsqueeze(1)
y_train_full = train_full.targets.clone().long()

x_test_full = test_full.data.clone().float().unsqueeze(1)
y_test_full = test_full.targets.clone().long()

print("Train:", x_train_full.shape, y_train_full.shape)
print("Test :", x_test_full.shape, y_test_full.shape)
print("Pixel range:", float(x_train_full.min()), "to", float(x_train_full.max()))

100%|█████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:01<00:00, 6.83MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 185kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 2.18MB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 2.28MB/s]


Train: torch.Size([60000, 1, 28, 28]) torch.Size([60000])
Test : torch.Size([10000, 1, 28, 28]) torch.Size([10000])
Pixel range: 0.0 to 255.0


In [4]:
def sample_mnist_split(
    x_train_all: torch.Tensor,
    y_train_all: torch.Tensor,
    x_test_all: torch.Tensor,
    y_test_all: torch.Tensor,
    n_train: int = N_TRAIN,
    n_val: int = N_VAL,
    n_test: int = N_TEST,
    seed: int = 0,
):
    rng = random.Random(seed)

    train_indices = torch.tensor(
        rng.sample(range(len(x_train_all)), n_train + n_val),
        dtype=torch.long,
    )
    test_indices = torch.tensor(
        rng.sample(range(len(x_test_all)), n_test),
        dtype=torch.long,
    )

    x_trainval = x_train_all[train_indices]
    y_trainval = y_train_all[train_indices]

    split = {
        "x_train": x_trainval[:n_train].clone(),
        "y_train": y_trainval[:n_train].clone(),
        "x_val": x_trainval[n_train:n_train + n_val].clone(),
        "y_val": y_trainval[n_train:n_train + n_val].clone(),
        "x_test": x_test_all[test_indices].clone(),
        "y_test": y_test_all[test_indices].clone(),
    }
    return split


def make_loaders(split: dict, batch_size: int = BATCH_SIZE):
    pin = device.type == "cuda"

    train_loader = DataLoader(
        TensorDataset(split["x_train"], split["y_train"]),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=pin,
    )
    val_loader = DataLoader(
        TensorDataset(split["x_val"], split["y_val"]),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin,
    )
    test_loader = DataLoader(
        TensorDataset(split["x_test"], split["y_test"]),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin,
    )
    return train_loader, val_loader, test_loader


example_split = sample_mnist_split(
    x_train_full, y_train_full, x_test_full, y_test_full, seed=BASE_SEED
)

for key, value in example_split.items():
    print(key, tuple(value.shape))

x_train (1200, 1, 28, 28)
y_train (1200,)
x_val (300, 1, 28, 28)
y_val (300,)
x_test (300, 1, 28, 28)
y_test (300,)


In [5]:
def init_like_keras(module: nn.Module):
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class H1CNN(nn.Module):
    def __init__(self, num_filters: int, kernel_size: int, num_classes: int = 10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, num_filters, kernel_size=kernel_size, stride=1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(num_filters, num_filters, kernel_size=kernel_size, stride=1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, 28, 28)
            flat_dim = self.features(dummy).flatten(1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 1024, bias=True),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(1024, num_classes, bias=True),
        )

        self.apply(init_like_keras)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


model = H1CNN(num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE).to(device)
print(model)

H1CNN(
  (features): Sequential(
    (0): Conv2d(1, 20, kernel_size=(4, 4), stride=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(20, 20, kernel_size=(4, 4), stride=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=320, out_features=1024, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=1024, out_features=10, bias=True)
  )
)


In [6]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_seen += xb.size(0)

    return total_loss / total_seen, total_correct / total_seen


def train_one_run(run_seed: int, kernel_size: int, num_filters: int, epochs: int = EPOCHS):
    seed_everything(run_seed)

    split = sample_mnist_split(
        x_train_full, y_train_full, x_test_full, y_test_full, seed=run_seed
    )
    train_loader, val_loader, test_loader = make_loaders(split, batch_size=BATCH_SIZE)

    model = H1CNN(num_filters=num_filters, kernel_size=kernel_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(BETA1, BETA2),
        eps=EPS,
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(dim=1) == yb).sum().item()
            running_seen += xb.size(0)

        train_loss = running_loss / running_seen
        train_acc = running_correct / running_seen
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

    test_loss, test_acc = evaluate(model, test_loader, criterion)

    return {
        "seed": run_seed,
        "history": pd.DataFrame(history),
        "test_loss": test_loss,
        "test_acc": test_acc,
    }

In [7]:
all_runs = []
all_histories = []

start = time.perf_counter()

for repeat_idx in tqdm(range(REPEATS), desc=f"H1 MNIST | k={KERNEL_SIZE}, F={NUM_FILTERS}"):
    run_seed = BASE_SEED + repeat_idx

    result = train_one_run(
        run_seed=run_seed,
        kernel_size=KERNEL_SIZE,
        num_filters=NUM_FILTERS,
        epochs=EPOCHS,
    )

    all_runs.append(
        {
            "repeat": repeat_idx + 1,
            "seed": result["seed"],
            "test_loss": result["test_loss"],
            "test_acc": result["test_acc"],
            "final_train_acc": result["history"]["train_acc"].iloc[-1],
            "final_val_acc": result["history"]["val_acc"].iloc[-1],
        }
    )
    all_histories.append(result["history"].assign(repeat=repeat_idx + 1))

elapsed_seconds = time.perf_counter() - start

runs_df = pd.DataFrame(all_runs)
histories_df = pd.concat(all_histories, ignore_index=True)

runs_df

H1 MNIST | k=4, F=20:   0%|          | 0/20 [00:00<?, ?it/s]

,repeat,seed,test_loss,test_acc,final_train_acc,final_val_acc
0,1,1234,7.141039,0.956667,0.998333,0.930000
1,2,1235,9.104174,0.950000,0.994167,0.923333
2,3,1236,9.254271,0.950000,0.998333,0.953333
3,4,1237,15.960761,0.913333,0.991667,0.926667
4,5,1238,6.873790,0.950000,0.996667,0.940000
5,6,1239,5.311594,0.940000,0.996667,0.936667
6,7,1240,7.041438,0.960000,0.994167,0.946667
7,8,1241,11.257484,0.950000,0.995000,0.916667
8,9,1242,5.908573,0.936667,0.995000,0.926667
9,10,1243,8.233676,0.943333,0.997500,0.920000


In [8]:
mean_acc = runs_df["test_acc"].mean()
std_acc = runs_df["test_acc"].std(ddof=1)


print(f"Config: H1 MNIST | kernel={KERNEL_SIZE}x{KERNEL_SIZE}, filters={NUM_FILTERS}")
print(f"Mean test accuracy over {REPEATS} runs : {mean_acc:.4f}")
print(f"Std. dev. over {REPEATS} runs         : {std_acc:.4f}")
print(f"Elapsed time                          : {elapsed_seconds / 60:.2f} minutes")

os.makedirs("./results", exist_ok=True)

runs_path = f"./results/mnist_h1_k{KERNEL_SIZE}_f{NUM_FILTERS}_runs.csv"
hist_path = f"./results/mnist_h1_k{KERNEL_SIZE}_f{NUM_FILTERS}_histories.csv"

runs_df.to_csv(runs_path, index=False)
histories_df.to_csv(hist_path, index=False)

print("\nSaved:")
print(runs_path)
print(hist_path)

Config: H1 MNIST | kernel=4x4, filters=20
Mean test accuracy over 20 runs : 0.9443
Std. dev. over 20 runs         : 0.0128
Elapsed time                          : 49.54 minutes

Saved:
./results/mnist_h1_k4_f20_runs.csv
./results/mnist_h1_k4_f20_histories.csv


In [9]:
def run_experiment_cell(kernel_size: int, num_filters: int, tag: str = "mnist_h1"):
    all_runs = []
    all_histories = []

    start = time.perf_counter()

    for repeat_idx in tqdm(
        range(REPEATS),
        desc=f"H1 MNIST | k={kernel_size}, F={num_filters}"
    ):
        run_seed = BASE_SEED + repeat_idx

        result = train_one_run(
            run_seed=run_seed,
            kernel_size=kernel_size,
            num_filters=num_filters,
            epochs=EPOCHS,
        )

        all_runs.append(
            {
                "repeat": repeat_idx + 1,
                "seed": result["seed"],
                "test_loss": result["test_loss"],
                "test_acc": result["test_acc"],
                "final_train_acc": result["history"]["train_acc"].iloc[-1],
                "final_val_acc": result["history"]["val_acc"].iloc[-1],
            }
        )
        all_histories.append(result["history"].assign(repeat=repeat_idx + 1))

    elapsed_seconds = time.perf_counter() - start

    runs_df = pd.DataFrame(all_runs)
    histories_df = pd.concat(all_histories, ignore_index=True)

    mean_acc = runs_df["test_acc"].mean()
    std_acc = runs_df["test_acc"].std(ddof=1)

    print(f"\nConfig: H1 MNIST | kernel={kernel_size}x{kernel_size}, filters={num_filters}")
    print(f"Mean test accuracy over {REPEATS} runs : {mean_acc:.4f}")
    print(f"Std. dev. over {REPEATS} runs         : {std_acc:.4f}")
    print(f"Elapsed time                          : {elapsed_seconds / 60:.2f} minutes")

    os.makedirs("./results", exist_ok=True)

    runs_path = f"./results/{tag}_k{kernel_size}_f{num_filters}_runs.csv"
    hist_path = f"./results/{tag}_k{kernel_size}_f{num_filters}_histories.csv"

    runs_df.to_csv(runs_path, index=False)
    histories_df.to_csv(hist_path, index=False)

    print("\nSaved:")
    print(runs_path)
    print(hist_path)

    return runs_df, histories_df

In [10]:
runs_k2_f1, histories_k2_f1 = run_experiment_cell(kernel_size=2, num_filters=1)

H1 MNIST | k=2, F=1:   0%|          | 0/20 [00:00<?, ?it/s]


Config: H1 MNIST | kernel=2x2, filters=1
Mean test accuracy over 20 runs : 0.7648
Std. dev. over 20 runs         : 0.1500
Elapsed time                          : 42.84 minutes

Saved:
./results/mnist_h1_k2_f1_runs.csv
./results/mnist_h1_k2_f1_histories.csv


In [11]:
runs_k2_f20, histories_k2_f20 = run_experiment_cell(kernel_size=2, num_filters=20)

H1 MNIST | k=2, F=20:   0%|          | 0/20 [00:00<?, ?it/s]


Config: H1 MNIST | kernel=2x2, filters=20
Mean test accuracy over 20 runs : 0.9263
Std. dev. over 20 runs         : 0.0209
Elapsed time                          : 51.24 minutes

Saved:
./results/mnist_h1_k2_f20_runs.csv
./results/mnist_h1_k2_f20_histories.csv


In [ ]:
runs_k2_f50, histories_k2_f50 = run_experiment_cell(kernel_size=2, num_filters=50)

In [ ]:
runs_k4_f1, histories_k4_f1 = run_experiment_cell(kernel_size=4, num_filters=1)

In [ ]:
runs_k4_f20, histories_k4_f20 = run_experiment_cell(kernel_size=4, num_filters=20)

In [ ]:
runs_k4_f50, histories_k4_f50 = run_experiment_cell(kernel_size=4, num_filters=50)